# Module 01: Feast Core Concepts

## What You'll Learn

- What problem a feature store solves (training-serving skew)
- Feast's core abstractions: Entity, Feature View, Data Source, Registry
- How Feast fits into the RHOAI platform
- How to define and apply a minimal feature store

## Prerequisites

- RHOAI workbench with `feast` installed
- `oc` CLI authenticated to your cluster
- A Data Science Project (namespace) with Feast operator available

---

> **🗺️ DATA STRATEGY**: Feast is the **predictive AI** data abstraction layer in the RHOAI data strategy (Pillar 3). It provides the feature store for online/offline feature serving — the foundation for real-time AI decisioning and model training with governed, consistent data.

## The Problem: Training-Serving Skew

The #1 cause of model performance degradation in production is **training-serving skew**: the features used to train a model differ from the features used to serve predictions.

This happens because:
1. Training uses batch SQL queries against a data warehouse
2. Serving uses a different code path (REST API, custom logic)
3. Transformations drift over time between the two paths

A feature store solves this by providing **one definition** of each feature that serves both training (offline) and inference (online).

```
WITHOUT Feature Store:              WITH Feature Store:

Training Path:                      Single Definition:
  SQL → pandas → train               FeatureView
                                        │
Serving Path:                           ├── get_historical_features() → train
  API → custom code → predict          └── get_online_features()     → predict
                                    
  ⚠️ These diverge silently!          ✅ Same transformation, same data
```

## Feast Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    Feast Feature Store                        │
│                                                              │
│  ┌──────────────┐                                           │
│  │   Registry   │ ← Stores metadata: what features exist,   │
│  │  (SQL/file)  │   their schemas, entities, sources        │
│  └──────────────┘                                           │
│                                                              │
│  ┌──────────────┐  ┌──────────────┐                        │
│  │ Offline Store │  │ Online Store  │                        │
│  │ (historical) │  │ (low-latency) │                        │
│  │              │  │               │                        │
│  │ PostgreSQL,  │  │ Redis, PG,    │                        │
│  │ Snowflake,   │  │ DynamoDB      │                        │
│  │ DuckDB, file │  │               │                        │
│  └──────────────┘  └──────────────┘                        │
│         │                  ▲                                 │
│         │    materialize   │                                 │
│         └──────────────────┘                                 │
│                                                              │
│  ┌──────────────┐                                           │
│  │Feature Server│ ← HTTP/gRPC API for online serving        │
│  └──────────────┘                                           │
└─────────────────────────────────────────────────────────────┘
```

Key components:
- **Registry**: metadata store — knows what features exist, their schemas, and relationships
- **Offline Store**: where historical data lives for training (point-in-time correct retrieval)
- **Online Store**: low-latency store for real-time inference
- **Materialization**: the process of moving data from offline → online store
- **Feature Server**: HTTP/gRPC endpoint for serving features to models

> **📍 RHOAI STATUS**: Feast is GA in RHOAI 3.4+ via the Feast Operator. The operator manages deployment of the registry, online store, offline store, feature server, and UI as a single `FeatureStore` Custom Resource.
>
> **⚠️ GAP**: The FeatureStore CRD is `v1alpha1` — API stability is not guaranteed between operator versions. This is a known P0 gap for enterprise adoption.

## Core Abstractions

### 1. Entity

An **Entity** is the thing you're computing features *about*. It defines the join key used to look up features.

Examples:
- A `customer` entity (join key: `customer_id`)
- A `transaction` entity (join key: `transaction_id`)
- A `driver` entity (join key: `driver_id`)

### 2. Data Source

A **Data Source** tells Feast where raw data lives. This is the input — the table, file, or stream that contains the raw feature values.

Types:
- `FileSource` — Parquet files (local or S3)
- `PostgreSQLSource` — PostgreSQL table/query
- `BigQuerySource`, `SnowflakeSource`, etc.
- `PushSource` — for streaming/real-time push (Alpha)

### 3. Feature View

A **Feature View** is the core abstraction — it defines:
- Which entity the features belong to
- Which data source provides the raw data
- Which columns are features (with types)
- TTL (time-to-live) for freshness

### 4. Feature Service

A **Feature Service** groups multiple feature views together for a specific use case (e.g., "all features needed for the fraud detection model").

### 5. Registry

The **Registry** stores all the metadata — the definitions of entities, feature views, data sources, and their relationships. It's the "catalog" of your feature store.

## Hands-On: Define a Minimal Feature Store

Let's define a simple feature store for a credit scoring use case. We'll create:
1. An entity (customer)
2. A data source (customer transaction stats)
3. A feature view (credit features)
4. Apply it to the registry

First, let's check our environment:

In [ ]:
import feast
print(f"Feast version: {feast.__version__}")

# Check if we're running in a workbench on RHOAI
import os
in_rhoai = os.path.exists('/opt/app-root')
print(f"Running in RHOAI workbench: {in_rhoai}")

### Step 1: Create sample data

In a real scenario, this data would come from your data warehouse or data lake. For learning, we'll generate a sample dataset.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Simulate customer credit features over time
np.random.seed(42)
n_customers = 100
n_days = 30

records = []
for customer_id in range(1, n_customers + 1):
    for day_offset in range(n_days):
        ts = datetime.now() - timedelta(days=day_offset)
        records.append({
            "customer_id": customer_id,
            "event_timestamp": ts,
            "credit_score": np.random.randint(300, 850),
            "account_age_days": np.random.randint(30, 3650),
            "num_transactions_30d": np.random.randint(0, 200),
            "avg_transaction_amount": round(np.random.uniform(10, 5000), 2),
            "missed_payments_12m": np.random.randint(0, 5),
        })

df = pd.DataFrame(records)
print(f"Generated {len(df)} records for {n_customers} customers over {n_days} days")
df.head()

In [ ]:
# Save as Parquet (the simplest data source for getting started)
import os
os.makedirs("data", exist_ok=True)
df.to_parquet("data/customer_credit_features.parquet")
print("Saved to data/customer_credit_features.parquet")

### Step 2: Define the Feature Store

Feast definitions live in Python files (by convention in a `feature_repo/` directory). Let's create them programmatically first, then we'll look at the file-based approach.

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Float32, Int64
from datetime import timedelta

# 1. Define the Entity — what we compute features about
customer = Entity(
    name="customer",
    join_keys=["customer_id"],
    description="A customer in the credit scoring system",
)

# 2. Define the Data Source — where raw data comes from
credit_source = FileSource(
    name="customer_credit_source",
    path=os.path.abspath("data/customer_credit_features.parquet"),
    timestamp_field="event_timestamp",
)

# 3. Define the Feature View — the features we serve
credit_features = FeatureView(
    name="customer_credit_features",
    entities=[customer],
    ttl=timedelta(days=1),  # Features older than 1 day are considered stale
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="account_age_days", dtype=Int64),
        Field(name="num_transactions_30d", dtype=Int64),
        Field(name="avg_transaction_amount", dtype=Float32),
        Field(name="missed_payments_12m", dtype=Int64),
    ],
    source=credit_source,
)

print("Defined: Entity(customer), FileSource(credit), FeatureView(credit_features)")
print(f"\nFeature View '{credit_features.name}':")
print(f"  Entity: {[e.name for e in credit_features.entities]}")
print(f"  TTL: {credit_features.ttl}")
print(f"  Fields: {[f.name for f in credit_features.schema]}")

### Step 3: Create the feature_store.yaml

The `feature_store.yaml` is the central configuration file. It tells Feast which backends to use.

For this first module, we'll use a local/simple configuration. In later modules (05, 12) we'll configure PostgreSQL, Redis, and the full operator-managed setup.

In [ ]:
import os
os.makedirs("feature_repo", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Write the feature_store.yaml configuration
with open("feature_repo/feature_store.yaml", "w") as f:
    f.write("""project: credit_scoring
provider: local
registry:
  registry_type: sql
  path: sqlite:///data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: duckdb
entity_key_serialization_version: 3
""")
print("Written: feature_repo/feature_store.yaml")

> **📍 RHOAI STATUS**: On RHOAI, you typically don't write `feature_store.yaml` manually. The Feast Operator generates it from the `FeatureStore` CR spec. But understanding this file is essential for debugging and for configuring features not yet exposed in the CRD (like OpenLineage, Ray compute, etc.).
>
> **⚠️ GAP**: The CRD does not expose all `feature_store.yaml` options. Advanced features (Ray compute, OpenLineage, MCP) require manual config outside the operator flow.

In [ ]:
# Write the feature definitions to a Python file (how Feast expects them)
os.makedirs("feature_repo", exist_ok=True)

feature_def = '''
from datetime import timedelta
import os

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

customer = Entity(
    name="customer",
    join_keys=["customer_id"],
    description="A customer in the credit scoring system",
)

credit_source = FileSource(
    name="customer_credit_source",
    path=os.path.abspath("../data/customer_credit_features.parquet"),
    timestamp_field="event_timestamp",
)

credit_features = FeatureView(
    name="customer_credit_features",
    entities=[customer],
    ttl=timedelta(days=1),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="account_age_days", dtype=Int64),
        Field(name="num_transactions_30d", dtype=Int64),
        Field(name="avg_transaction_amount", dtype=Float32),
        Field(name="missed_payments_12m", dtype=Int64),
    ],
    source=credit_source,
)
'''

with open("feature_repo/credit_features.py", "w") as f:
    f.write(feature_def)

print("Written: feature_repo/credit_features.py")
print("Written: feature_repo/feature_store.yaml")

### Step 4: Apply — Register Features in the Registry

`feast apply` is the command that reads your Python definitions and registers them in the registry. It's analogous to `kubectl apply` — it makes the desired state real.

In [ ]:
# Apply the feature definitions to the registry
store = FeatureStore(repo_path="feature_repo")
store.apply([customer, credit_source, credit_features])

print("✅ Applied feature definitions to registry")
print(f"\nRegistered feature views: {[fv.name for fv in store.list_feature_views()]}")
print(f"Registered entities: {[e.name for e in store.list_entities()]}")
print(f"Registered data sources: {[ds.name for ds in store.list_data_sources()]}")

> **🗺️ DATA STRATEGY**: `feast apply` is where OpenLineage events are first emitted (Module 07). When lineage is enabled, this command creates the lineage graph edges: data source → feature view → online store. This is how Feast contributes to the E2E lineage story (Workshop Decision #1).

### Step 5: Verify — Browse the Registry

Let's inspect what's in the registry now.

In [ ]:
# Inspect the registered feature view
fv = store.get_feature_view("customer_credit_features")

print(f"Feature View: {fv.name}")
print(f"  Entities: {[e.name for e in fv.entities]}")
print(f"  Source: {fv.batch_source.name}")
print(f"  TTL: {fv.ttl}")
print(f"  Schema:")
for field in fv.schema:
    print(f"    - {field.name}: {field.dtype}")

> **🖥️ UI**: On RHOAI, you can also browse this information visually in the **Feast UI**. If the operator deployed it, access via:
> ```bash
> oc get routes -n <your-namespace> | grep feast-ui
> ```
> The UI shows the same entities, feature views, and data sources in a web interface. See [docs/ui-guide.md](../../docs/ui-guide.md) for details.
>
> **⚠️ GAP**: The Feast UI is Beta maturity. It's read-only — you cannot modify definitions or permissions from the UI.

## Quick Test: Retrieve Features

Let's do a quick retrieval to prove the system works. We'll cover this in depth in Modules 02 (offline) and 03 (online).

In [ ]:
# Retrieve historical features for a few customers
entity_df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "event_timestamp": [datetime.now()] * 5,
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_credit_features:credit_score",
        "customer_credit_features:num_transactions_30d",
        "customer_credit_features:avg_transaction_amount",
    ],
).to_df()

print("Historical feature retrieval (offline store):")
training_df

## Key Takeaways

1. **Feature stores solve training-serving skew** — one definition serves both training and inference
2. **Feast has 5 core abstractions**: Entity, Data Source, Feature View, Feature Service, Registry
3. **`feast apply`** registers definitions in the registry (and emits OpenLineage events)
4. **Offline store** = historical data for training; **Online store** = low-latency for inference
5. **Materialization** moves data from offline → online (covered in Module 03)

## What's Next

- **Module 02**: Deep dive into offline store — point-in-time joins, entity DataFrames, training datasets
- **Module 03**: Online store — materialization, `get_online_features()`, TTL and freshness

## Further Reading

- [Feast Concepts Documentation](https://docs.feast.dev/getting-started/concepts)
- [Feast Architecture](https://docs.feast.dev/getting-started/architecture)
- [Feast on RHOAI Blog Post](https://www.redhat.com/en/blog/feast-open-source-feature-store-ai)